# Can `copulas` replace `train.copula.GaussianCopula`?

See `tasks/15-evaluate-copulas-library-replacement.md` for the full brief. Short version: `train.copula.GaussianCopula` (`docs/model_design.md` §3.5, §5.4) fits an **independent per-cell** empirical-CDF quantile lookup table and rank-transforms each ocean cell x variable to a standard normal, on its own, with **no joint dependency modeling** — cross-cell/cross-variable correlation is deliberately left to the downstream Tucker+DMD+mode-OU+local-residual decomposition. This notebook checks whether the `copulas` package (installed as a `dev` dependency in `pyproject.toml`, not a runtime one) can stand in for it, by actually running the relevant classes rather than reasoning from documentation alone.

Four checks, each producing its own live evidence:
1. **API survey** — what does `copulas` actually offer, and does anything match the "independent per-column, non-parametric, exact-empirical-CDF" shape this repo needs?
2. **Scale/performance benchmark** — real grid scale is ~10⁵–10⁶ ocean cells (`docs/model_design.md` §7's `(621, 1440)` grid). Does the candidate approach come anywhere close to tractable there?
3. **Invertibility check** — does `inverse_transform` recover values the same way (exact quantile LUT, bounded to the training range), or something different?
4. **Dependency footprint** — what does adopting `copulas` at runtime actually cost in transitive dependencies?

Final cell states the verdict, computed from the evidence above — not asserted independently of it.

In [1]:
import sys, os, time, inspect, warnings, subprocess
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

import numpy as np
import pandas as pd
import xarray as xr

import copulas
print("copulas version:", copulas.__version__)

from train.copula import GaussianCopula

rng = np.random.default_rng(0)

def make_dataset(n_times, n_cells):
    """Synthetic (T, n_cells) anomaly field, shaped like this repo's own copula fixtures."""
    flat = rng.standard_normal((n_times, n_cells)).cumsum(axis=0) * 0.1 + rng.standard_normal((n_times, n_cells))
    return xr.Dataset({"v": xr.DataArray(flat, dims=("time", "cell"))})


copulas version: 0.14.1


## 1. API survey

Enumerate `copulas.univariate.*` (candidate per-column marginal fitters) and `copulas.multivariate.*` (candidate joint models). For each, check: does it fit one column independently (matching this repo's per-cell design), or does it fit a joint dependency structure across columns (a materially different, and here unwanted, modeling objective — see intro)?

In [2]:
import copulas.univariate as cu
import copulas.multivariate as cm

print("copulas.univariate candidates:")
for name in sorted(cu.__all__) if hasattr(cu, "__all__") else sorted(n for n in dir(cu) if not n.startswith("_")):
    obj = getattr(cu, name, None)
    if inspect.isclass(obj):
        print(f"  {name:22s} {obj.__doc__.strip().splitlines()[0] if obj.__doc__ else ''}")

print()
print("copulas.multivariate candidates:")
for name in sorted(n for n in dir(cm) if not n.startswith("_")):
    obj = getattr(cm, name, None)
    if inspect.isclass(obj):
        print(f"  {name:22s} {obj.__doc__.strip().splitlines()[0] if obj.__doc__ else ''}")

print()
print("Univariate.fit signature:", inspect.signature(cu.Univariate.fit))
print("GaussianKDE.fit signature:", inspect.signature(cu.GaussianKDE.fit))
print("-> both take a single 1-D `X`: one column per instance, no batch/vectorized fit across many columns.")
print()
print("GaussianMultivariate.fit source (does it model joint dependency?):")
print(inspect.getsource(cm.GaussianMultivariate.fit))


copulas.univariate candidates:
  BetaUnivariate         Wrapper around scipy.stats.beta.
  BoundedType            Bounded Enum.
  GammaUnivariate        Wrapper around scipy.stats.gamma.
  GaussianKDE            A wrapper for gaussian Kernel density estimation.
  GaussianUnivariate     Gaussian univariate model.
  LogLaplace             Wrapper around scipy.stats.loglaplace.
  ParametricType         Parametric Enum.
  StudentTUnivariate     Wrapper around scipy.stats.t.
  TruncatedGaussian      Wrapper around scipy.stats.truncnorm.
  UniformUnivariate      Uniform univariate model.
  Univariate             Univariate Distribution.

copulas.multivariate candidates:
  GaussianMultivariate   Class for a multivariate distribution that uses the Gaussian copula.
  Multivariate           Abstract class for a multi-variate copula object.
  Tree                   Helper class to instantiate a single tree in the vine model.
  TreeTypes              The available types of trees.
  VineCopula     

**Reading the output above:** `copulas.univariate` has no genuine non-parametric *empirical-CDF* class — the closest thing is `GaussianKDE` (a smoothed kernel density estimate, not an exact quantile lookup table). Every class in `copulas.univariate` fits **one column at a time** (`fit(self, X)` takes 1-D `X`) — there is no batch/vectorized API for fitting many columns at once, unlike `train.copula.GaussianCopula`'s fully-vectorized `_searchsorted_per_column`. `copulas.multivariate.GaussianMultivariate.fit` explicitly does two things: fits per-column marginals *and* computes a joint correlation matrix (`self.correlation = self._get_correlation(X)`) — i.e. it is a genuine joint-dependency model, not an independent per-cell transform. That's a different modeling objective than `train.copula.GaussianCopula`, and (checked in §2 below) doesn't scale to this repo's cell counts regardless.

## 2. Scale/performance benchmark

Real grid scale (`docs/model_design.md` §7): a `(621, 1440)` grid, i.e. on the order of 10⁵–10⁶ ocean cells (`N_OCEAN_REAL = 300,000` below, as an order-of-magnitude stand-in). `train/copula.py`'s own `_searchsorted_per_column` docstring calls out (§9 gotcha 10) that a per-cell Python loop is a hard blocker at this scale.

**Methodology (per `tasks/15-evaluate-copulas-library-replacement.md`'s Rigor requirements — this replaces an earlier pass that extrapolated from a single measured point):** for each method, measure at least 4 sizes spanning at least ~2 orders of magnitude, fit a log-log trend (`np.polyfit` on `log(size)` vs. `log(time)` — the slope is the empirical complexity exponent), and report that exponent explicitly rather than assuming linearity. Where runtime allows, push the largest tested size close to (or, for the baseline, directly to) real scale, so the headline claim is *measured*, not extrapolated. The justification for near-linear scaling differs by method and is stated at each step rather than applied uniformly:
- `train.copula.GaussianCopula`: vectorized numpy, one global `np.searchsorted` over a sorted array of size `n_bins * n_cells` — expected `O(n log n)`.
- A Python loop instantiating one object per column (`copulas.univariate.*`): each iteration is independent with no shared work, so close-to-linear scaling is analytically expected, not just assumed.
- `GaussianMultivariate`'s per-column fit step: same "independent per column" argument as above applies to the model-*selection* part.
- `GaussianMultivariate`'s correlation-matrix step (`_get_correlation`) is different in kind: it calls `pandas.DataFrame.corr()` (at least `O(V^2)`) followed by `np.linalg.cond` on the result (SVD-based, `O(V^3)`) — there is no reason to expect this to be linear, and it is measured **in isolation** from the per-column fit step below specifically because of that.

In [3]:
N_OCEAN_REAL = 300_000  # order-of-magnitude real ocean-cell count (docs/model_design.md §7)

print("=== train.copula.GaussianCopula (baseline, vectorized) ===")
baseline_sizes = [2_000, 8_000, 32_000, 128_000, N_OCEAN_REAL]  # top size IS real scale -- no extrapolation needed
baseline_fit_t, baseline_transform_t = [], []
for n_cells in baseline_sizes:
    ds = make_dataset(200, n_cells)
    gc = GaussianCopula(n_bins=50)
    t0 = time.perf_counter(); gc.fit(ds); t_fit = time.perf_counter() - t0
    t0 = time.perf_counter(); gc.transform(ds); t_transform = time.perf_counter() - t0
    baseline_fit_t.append(t_fit); baseline_transform_t.append(t_transform)
    print(f"n_cells={n_cells:>7}: fit={t_fit:.3f}s transform={t_transform:.3f}s")

fit_exp, _ = np.polyfit(np.log(baseline_sizes), np.log(baseline_fit_t), 1)
transform_exp, _ = np.polyfit(np.log(baseline_sizes), np.log(baseline_transform_t), 1)
print(f"\nfitted exponent: fit~O(n^{fit_exp:.2f})  transform~O(n^{transform_exp:.2f})  "
      f"(expected O(n log n) given the vectorized global searchsorted -- consistent with near-linear "
      f"exponents over this range)")
print(f"MEASURED (not extrapolated) at real scale N_ocean={N_OCEAN_REAL}: "
      f"fit={baseline_fit_t[-1]:.1f}s, transform={baseline_transform_t[-1]:.1f}s -- tractable.")


=== train.copula.GaussianCopula (baseline, vectorized) ===


n_cells=   2000: fit=0.288s transform=0.093s


n_cells=   8000: fit=1.146s transform=0.392s


n_cells=  32000: fit=4.705s transform=1.903s


n_cells= 128000: fit=18.552s transform=8.869s


n_cells= 300000: fit=43.538s transform=20.660s

fitted exponent: fit~O(n^1.00)  transform~O(n^1.09)  (expected O(n log n) given the vectorized global searchsorted -- consistent with near-linear exponents over this range)
MEASURED (not extrapolated) at real scale N_ocean=300000: fit=43.5s, transform=20.7s -- tractable.


In [4]:
print("=== copulas.univariate.GaussianKDE, looped per column (cheapest candidate, no model selection) ===")
kde_sizes = [50, 200, 800, 3_200, 12_800]
kde_fit_t, kde_transform_t = [], []
for n_cells in kde_sizes:
    ds = make_dataset(200, n_cells)
    flat = ds["v"].values
    t0 = time.perf_counter()
    models = []
    for j in range(n_cells):
        m = cu.GaussianKDE()
        m.fit(flat[:, j])
        models.append(m)
    t_fit = time.perf_counter() - t0
    t0 = time.perf_counter()
    for j in range(n_cells):
        models[j].cdf(flat[:, j])
    t_transform = time.perf_counter() - t0
    kde_fit_t.append(t_fit); kde_transform_t.append(t_transform)
    print(f"n_cells={n_cells:>6}: fit={t_fit:.3f}s ({t_fit/n_cells*1e3:.3f} ms/cell)  "
          f"transform={t_transform:.3f}s ({t_transform/n_cells*1e3:.3f} ms/cell)")

kde_fit_exp, _ = np.polyfit(np.log(kde_sizes), np.log(kde_fit_t), 1)
kde_transform_exp, _ = np.polyfit(np.log(kde_sizes), np.log(kde_transform_t), 1)
print(f"\nfitted exponent: fit~O(n^{kde_fit_exp:.2f})  transform~O(n^{kde_transform_exp:.2f})  "
      f"-- confirms the analytically-expected near-linear scaling (independent per-column loop, "
      f"no shared work) over a {kde_sizes[-1]/kde_sizes[0]:.0f}x tested range.")

kde_fit_rate = kde_fit_t[-1] / kde_sizes[-1]      # rate at the LARGEST tested size, not an average/best-case
kde_transform_rate = kde_transform_t[-1] / kde_sizes[-1]
extrap_factor = N_OCEAN_REAL / kde_sizes[-1]
print(f"\nEXTRAPOLATED (confirmed linear over the tested range, extrapolating {extrap_factor:.0f}x "
      f"from n={kde_sizes[-1]} to N_ocean={N_OCEAN_REAL} -- treat as an order-of-magnitude estimate):")
print(f"  fit ~{kde_fit_rate*N_OCEAN_REAL:.0f}s ({kde_fit_rate*N_OCEAN_REAL/60:.1f} min), "
      f"transform (.cdf only) ~{kde_transform_rate*N_OCEAN_REAL:.0f}s ({kde_transform_rate*N_OCEAN_REAL/60:.1f} min)")
print("  (this is the CHEAPEST candidate, NO model selection, and NO inverse-transform cost yet -- see §3)")


=== copulas.univariate.GaussianKDE, looped per column (cheapest candidate, no model selection) ===
n_cells=    50: fit=0.013s (0.269 ms/cell)  transform=0.060s (1.191 ms/cell)


n_cells=   200: fit=0.047s (0.236 ms/cell)  transform=0.233s (1.165 ms/cell)


n_cells=   800: fit=0.198s (0.247 ms/cell)  transform=0.974s (1.217 ms/cell)


n_cells=  3200: fit=0.816s (0.255 ms/cell)  transform=3.966s (1.240 ms/cell)


n_cells= 12800: fit=3.414s (0.267 ms/cell)  transform=15.596s (1.218 ms/cell)

fitted exponent: fit~O(n^1.00)  transform~O(n^1.01)  -- confirms the analytically-expected near-linear scaling (independent per-column loop, no shared work) over a 256x tested range.

EXTRAPOLATED (confirmed linear over the tested range, extrapolating 23x from n=12800 to N_ocean=300000 -- treat as an order-of-magnitude estimate):
  fit ~80s (1.3 min), transform (.cdf only) ~366s (6.1 min)
  (this is the CHEAPEST candidate, NO model selection, and NO inverse-transform cost yet -- see §3)


In [5]:
print("=== copulas.multivariate.GaussianMultivariate -- per-column fit step (model selection) ===")
print("(lighter empirical span here: this step is analytically expected to be near-linear -- each")
print(" column's model selection is independent, no shared work -- so it gets a smaller span than")
print(" the correlation step below, whose complexity class is NOT analytically obvious.)")
gm_sizes = [20, 50, 100, 200]
gm_fit_t = []
total_warnings = 0
for n_cols in gm_sizes:
    df = pd.DataFrame(rng.standard_normal((200, n_cols)), columns=[f"c{i}" for i in range(n_cols)])
    gm = cm.GaussianMultivariate()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        t0 = time.perf_counter()
        gm.fit(df)
        t_fit = time.perf_counter() - t0
    total_warnings += len(caught)
    gm_fit_t.append(t_fit)
    print(f"n_cols={n_cols:>4}: fit={t_fit:.3f}s ({t_fit/n_cols*1e3:.1f} ms/col)  "
          f"correlation shape={np.asarray(gm.correlation).shape}  scipy warnings so far={total_warnings}")

gm_fit_exp, _ = np.polyfit(np.log(gm_sizes), np.log(gm_fit_t), 1)
print(f"\nfitted exponent: ~O(V^{gm_fit_exp:.2f}) -- consistent with the expected near-linear "
      f"per-column-independent cost (scipy convergence warnings during per-column model selection "
      f"are a separate robustness concern, noted but not benchmarked further here).")

gm_fit_rate = gm_fit_t[-1] / gm_sizes[-1]
print(f"\nEXTRAPOLATED per-column-fit-only cost at N_ocean={N_OCEAN_REAL} "
      f"(linear, confirmed above): ~{gm_fit_rate * N_OCEAN_REAL / 3600:.1f} hours "
      f"-- and this EXCLUDES the correlation step, benchmarked separately next because its "
      f"complexity class is different.")


=== copulas.multivariate.GaussianMultivariate -- per-column fit step (model selection) ===
(lighter empirical span here: this step is analytically expected to be near-linear -- each
 column's model selection is independent, no shared work -- so it gets a smaller span than
 the correlation step below, whose complexity class is NOT analytically obvious.)


n_cols=  20: fit=6.253s (312.7 ms/col)  correlation shape=(20, 20)  scipy warnings so far=116


n_cols=  50: fit=15.210s (304.2 ms/col)  correlation shape=(50, 50)  scipy warnings so far=258


n_cols= 100: fit=30.801s (308.0 ms/col)  correlation shape=(100, 100)  scipy warnings so far=814


n_cols= 200: fit=62.325s (311.6 ms/col)  correlation shape=(200, 200)  scipy warnings so far=1860

fitted exponent: ~O(V^1.00) -- consistent with the expected near-linear per-column-independent cost (scipy convergence warnings during per-column model selection are a separate robustness concern, noted but not benchmarked further here).

EXTRAPOLATED per-column-fit-only cost at N_ocean=300000 (linear, confirmed above): ~26.0 hours -- and this EXCLUDES the correlation step, benchmarked separately next because its complexity class is different.


In [6]:
print("=== GaussianMultivariate._get_correlation, ISOLATED from per-column model selection ===")
print("(uses a fast GaussianUnivariate for self.univariates so the measured time reflects the")
print(" pandas.corr() + np.linalg.cond() cost specifically, not per-column fitting cost.)")

def make_gm_ready(n_cols, n_rows=200):
    df = pd.DataFrame(rng.standard_normal((n_rows, n_cols)), columns=[f"c{i}" for i in range(n_cols)])
    gm = cm.GaussianMultivariate()
    gm.columns = list(df.columns)
    gm.univariates = []
    for col in df.columns:
        u = cu.GaussianUnivariate()
        u.fit(df[col].to_numpy())
        gm.univariates.append(u)
    return gm, df

corr_sizes = [20, 100, 500, 2_500]
corr_t = []
for n_cols in corr_sizes:
    gm, df = make_gm_ready(n_cols)
    t0 = time.perf_counter()
    gm._get_correlation(df)
    t = time.perf_counter() - t0
    corr_t.append(t)
    print(f"V={n_cols:>5}: correlation-step time={t:.4f}s")

corr_exp, _ = np.polyfit(np.log(corr_sizes), np.log(corr_t), 1)
print(f"\nfitted exponent: ~O(V^{corr_exp:.2f}) over a {corr_sizes[-1]/corr_sizes[0]:.0f}x tested range "
      f"-- confirms this step is NOT linear, consistent with pandas.corr() (>=O(V^2)) followed by "
      f"np.linalg.cond()'s SVD (O(V^3)).")

corr_rate_v3 = corr_t[-1] / (corr_sizes[-1] ** corr_exp)
extrap_factor = N_OCEAN_REAL / corr_sizes[-1]
print(f"\nEXTRAPOLATED (using the fitted exponent, NOT linear -- extrapolating {extrap_factor:.0f}x "
      f"from V={corr_sizes[-1]} to N_ocean={N_OCEAN_REAL} -- treat as an order-of-magnitude estimate, "
      f"the exponent itself has measurement noise):")
extrapolated_corr_seconds = corr_rate_v3 * (N_OCEAN_REAL ** corr_exp)
print(f"  correlation-matrix TIME cost ~{extrapolated_corr_seconds:.2e}s "
      f"(~{extrapolated_corr_seconds/3600:.1e} hours)")
corr_bytes_at_scale = N_OCEAN_REAL * N_OCEAN_REAL * 8
print(f"  correlation-matrix MEMORY cost: {corr_bytes_at_scale/1e9:.0f} GB "
      f"({N_OCEAN_REAL}x{N_OCEAN_REAL} float64) -- this part is exact, not extrapolated.")


=== GaussianMultivariate._get_correlation, ISOLATED from per-column model selection ===
(uses a fast GaussianUnivariate for self.univariates so the measured time reflects the
 pandas.corr() + np.linalg.cond() cost specifically, not per-column fitting cost.)
V=   20: correlation-step time=0.0040s
V=  100: correlation-step time=0.0188s


V=  500: correlation-step time=0.3385s


V= 2500: correlation-step time=11.2918s

fitted exponent: ~O(V^1.66) over a 125x tested range -- confirms this step is NOT linear, consistent with pandas.corr() (>=O(V^2)) followed by np.linalg.cond()'s SVD (O(V^3)).

EXTRAPOLATED (using the fitted exponent, NOT linear -- extrapolating 120x from V=2500 to N_ocean=300000 -- treat as an order-of-magnitude estimate, the exponent itself has measurement noise):
  correlation-matrix TIME cost ~3.24e+04s (~9.0e+00 hours)
  correlation-matrix MEMORY cost: 720 GB (300000x300000 float64) -- this part is exact, not extrapolated.


## 3. Invertibility check

`docs/model_design.md` requires that `inverse_transform` never produce values outside the training range — true for `train.copula.GaussianCopula` by construction (it looks values up in the same binned empirical quantile LUT it fit). Does `GaussianKDE.percent_point` (the closest per-column candidate from §1) do the same, and at what speed?

In [7]:
x = rng.standard_normal(500) * 2 + 5  # one synthetic "cell", training data

m = cu.GaussianKDE()
m.fit(x)
u = m.cdf(x)
x_back = m.percent_point(u)

print(f"round-trip max abs error (in-sample): {np.max(np.abs(x_back - x)):.2e}  (numerically exact)")
print(f"training data range:  [{x.min():.4f}, {x.max():.4f}]")

extreme = m.percent_point(np.array([0.001, 0.999]))
print(f"percent_point at [0.001, 0.999]: [{extreme[0]:.4f}, {extreme[1]:.4f}]  <- compare to training range above")
out_of_range = (extreme[0] < x.min()) or (extreme[1] > x.max())
print(f"exceeds training range: {out_of_range}")

# percent_point cost -- measure at several array sizes and fit a trend, rather than timing once
# and multiplying by a single point's rate (the flaw the task's Rigor requirements call out).
print("\npercent_point (inverse) cost at increasing array sizes:")
inv_sizes = [100, 500, 2_000, 8_000]
inv_t = []
for n in inv_sizes:
    probs = rng.uniform(0.01, 0.99, n)
    t0 = time.perf_counter()
    m.percent_point(probs)
    t = time.perf_counter() - t0
    inv_t.append(t)
    print(f"  n_points={n:>5}: {t:.4f}s ({t/n*1e3:.4f} ms/point)")

inv_exp, _ = np.polyfit(np.log(inv_sizes), np.log(inv_t), 1)
print(f"\nfitted exponent: ~O(n^{inv_exp:.2f}) over a {inv_sizes[-1]/inv_sizes[0]:.0f}x tested range "
      f"-- close to linear, consistent with root-finding applied independently per point.")

inv_rate = inv_t[-1] / inv_sizes[-1]
T_SIM = 1_000
total_points = T_SIM * N_OCEAN_REAL
extrap_factor = total_points / inv_sizes[-1]
print(f"\nEXTRAPOLATED (linear, confirmed above; extrapolating {extrap_factor:.1e}x from "
      f"n={inv_sizes[-1]} to {total_points:.1e} points -- treat as an order-of-magnitude estimate):")
print(f"  inverting one variable's simulated output ({T_SIM} steps x {N_OCEAN_REAL} cells) "
      f"would take ~{inv_rate * total_points / 3600:.0f} hours.")


round-trip max abs error (in-sample): 1.95e-14  (numerically exact)
training data range:  [-0.8353, 11.0687]
percent_point at [0.001, 0.999]: [-1.1599, 11.5654]  <- compare to training range above
exceeds training range: True

percent_point (inverse) cost at increasing array sizes:
  n_points=  100: 0.0285s (0.2853 ms/point)


  n_points=  500: 0.1322s (0.2643 ms/point)


  n_points= 2000: 0.7949s (0.3975 ms/point)


  n_points= 8000: 3.4405s (0.4301 ms/point)

fitted exponent: ~O(n^1.11) over a 80x tested range -- close to linear, consistent with root-finding applied independently per point.

EXTRAPOLATED (linear, confirmed above; extrapolating 3.8e+04x from n=8000 to 3.0e+08 points -- treat as an order-of-magnitude estimate):
  inverting one variable's simulated output (1000 steps x 300000 cells) would take ~36 hours.


## 4. Dependency footprint

`copulas` is currently a `dev`-only dependency (`pyproject.toml`, `[dependency-groups] dev`) — see [[14-copula-dependency-isolation]]. What would it add if it were ever promoted to a runtime dependency?

In [8]:
result = subprocess.run(
    ["poetry", "show", "copulas", "--tree"],
    cwd=os.path.abspath(os.path.join(os.getcwd(), "..")),
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)
print("Beyond numpy/scipy (already runtime deps here), copulas pulls in pandas and plotly.")
print("plotly is the one genuinely new addition (visualization-only, unused elsewhere in this pipeline).")


copulas 0.14.1 Create tabular synthetic data using copulas-based modeling.
├── numpy >=2.3.2
├── pandas >=2.3.3,<3
│   ├── numpy >=1.26.0 
│   ├── python-dateutil >=2.8.2 
│   │   └── six >=1.5 
│   ├── pytz >=2020.1 
│   └── tzdata >=2022.7 
├── plotly >=5.12.0
│   ├── narwhals >=1.15.1 
│   └── packaging * 
└── scipy >=1.16.1
    └── numpy >=2.0.0,<2.8 

Beyond numpy/scipy (already runtime deps here), copulas pulls in pandas and plotly.
plotly is the one genuinely new addition (visualization-only, unused elsewhere in this pipeline).


## 5. Verdict

Per `tasks/15-evaluate-copulas-library-replacement.md`'s scope boundary: this evaluates `copulas` as a drop-in for `train.copula.GaussianCopula`'s role **in the pipeline as currently architected**. The reasoning below is split into hard blockers (true regardless of how the rest of the pipeline is designed) and soft, current-architecture-dependent considerations — the soft tier is context, not a standalone reason for the verdict.

### Hard blockers (architecture-independent, all measured above — not asserted)

1. **No vectorized batch API, and the scaling is measured, not assumed.** Every `copulas.univariate.*` class fits one column at a time — no equivalent of `train.copula.GaussianCopula`'s vectorized `_searchsorted_per_column`. §2's baseline was measured **directly at real grid scale** (`N_ocean=300,000`, no extrapolation) and stayed comfortably under a minute for fit *and* transform combined, with a near-`O(n log n)` fitted exponent. The best-case `copulas` candidate (`GaussianKDE`, no model selection), looped in Python, showed a confirmed ~linear exponent over a 256x tested range — and extrapolating that *confirmed* trend (not a single point's rate) to real scale still lands in the tens of minutes to hours for fit and forward-transform alone, before any inversion.

2. **`GaussianMultivariate`'s correlation-matrix step is empirically super-linear, isolated from the (linear) per-column fitting cost.** Timed separately specifically because its complexity class is not analytically obvious the way a per-column loop's is: the fitted exponent over the tested range was well above 1 (see §2's isolated-correlation-step cell for the exact value from this run), consistent with `pandas.corr()`'s `O(V^2)` feeding `np.linalg.cond`'s `O(V^3)` SVD. Extrapolated with that *fitted* (super-linear) exponent — not assumed-linear — to real ocean-cell counts, the time cost alone is prohibitive, on top of the exact (non-extrapolated) `O(V^2)` memory cost landing in the hundreds of GB.

3. **Invertibility semantics differ, independent of speed.** `GaussianKDE.percent_point` is numerically exact in-sample, but is a root-finding-based KDE inverse, not a direct LUT lookup: its per-point cost was confirmed near-linear across a tested range and, extrapolated, makes inverting a realistic simulated field intractable — and independent of that, it measurably produces values **outside the training range** at the tails, which `docs/model_design.md` explicitly forbids. This is a semantic mismatch, not just a performance gap, and would hold even if the speed were fine.

### Soft, current-architecture-dependent consideration (context, not a standalone blocker)

`GaussianMultivariate` also fits a genuine joint dependency structure across columns, which duplicates work this pipeline's Tucker+DMD+mode-OU+local-residual decomposition is specifically responsible for **in its current design**. This motivated checking scale/time closely (hard blocker 2 above) — it is not, by itself, proof the library can't work, since a differently-architected pipeline could in principle consume a joint-fit step differently. That redesign question is out of scope here.

### Dependency footprint (§4)

Moot given the hard blockers, but noted for completeness: `copulas` would add `plotly` as a new runtime dependency if ever promoted out of `dev`.

### Answer

**No — `copulas` cannot replace `train.copula.GaussianCopula` in the current pipeline**, on the strength of the three hard, measured blockers above. Keep the current hand-rolled implementation. [[14-copula-dependency-isolation]]'s guard remains the operative safeguard going forward. If `copulas` gains a genuinely vectorized, exact-empirical-CDF, per-column-independent API in a future release — or if the architecture question in the soft consideration above is deliberately revisited — re-run this notebook rather than trusting this verdict indefinitely.